# 02 - Baseline Model

Purpose: train simple baseline regressors for SOH and RUL and establish reference performance.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Load feature table

In [2]:
from src.train import load_data
from src.models import prepare_cross_battery_data
from src.evaluation import regression_metrics
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

feature_df = load_data(PROCESSED_DIR / 'features_all.csv')
feature_df.head()

,battery_id,cycle_number,capacity_Ah,avg_voltage,avg_current,avg_temperature,charge_time,discharge_time,energy_charged_Wh,SOH,...,max_temperature,temperature_rise,discharge_duration,energy_discharged,internal_resistance_proxy,capacity_fade_rate,cycle_efficiency,capacity_lag_1,capacity_lag_3,capacity_lag_5
0,B0005,1,1.862203,3.529829,1.818712,32.572328,7597.875,3690.234,3.276268,93.110144,...,38.982181,14.652147,3690.234,6.608778,0.216948,-0.004175,2.017167,1.862203,1.862203,1.862203
1,B0005,2,1.852078,3.537320,1.817644,32.725235,10516.000,3672.344,7.638814,92.603889,...,39.033398,14.335646,3672.344,6.586345,0.990131,-0.004175,0.862221,1.862203,1.862203,1.862203
2,B0005,3,1.841049,3.543737,1.816542,32.642862,10484.547,3651.641,7.608577,92.052437,...,38.818797,14.084531,3651.641,6.555683,26.274884,-0.004175,0.861618,1.852078,1.862203,1.862203
3,B0005,4,1.840912,3.543666,1.825618,32.514876,10397.890,3631.563,7.578063,92.045600,...,38.762305,14.108068,3631.563,6.554829,0.235616,-0.004175,0.864974,1.841049,1.862203,1.862203
4,B0005,5,1.840360,3.542343,1.826148,32.382349,10495.203,3629.172,7.565826,92.017996,...,38.665393,14.140596,3629.172,6.552232,0.094378,-0.004175,0.866030,1.840912,1.852078,1.862203


## Train baseline models

In [3]:
records = []

# SOH baselines
soh_prepared = prepare_cross_battery_data(feature_df, target_col='SOH', test_battery='B0018')
for model_name, model in [
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression', Ridge(alpha=1.0, random_state=42)),
    ('Random Forest Regressor', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
]:
    model.fit(soh_prepared.X_train, soh_prepared.y_train)
    pred = model.predict(soh_prepared.X_test)
    m = regression_metrics(soh_prepared.y_test, pred)
    records.append({'Model': model_name, 'Task': 'SOH', **m})

# RUL baseline
rul_prepared = prepare_cross_battery_data(feature_df, target_col='RUL', test_battery='B0018')
rul_model = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rul_model.fit(rul_prepared.X_train, rul_prepared.y_train)
rul_pred = rul_model.predict(rul_prepared.X_test)
rul_m = regression_metrics(rul_prepared.y_test, rul_pred)
records.append({'Model': 'Random Forest Regressor', 'Task': 'RUL', **rul_m})

pd.DataFrame(records).sort_values(['Task','RMSE']).reset_index(drop=True)

,Model,Task,RMSE,MAE,MAPE,R2
0,Random Forest Regressor,RUL,6.373208e+00,5.091485e+00,1.797304e+01,0.962434
1,Linear Regression,SOH,2.333771e-14,1.700996e-14,2.024057e-14,1.000000
2,Random Forest Regressor,SOH,1.617960e-01,9.245272e-02,1.154639e-01,0.999552
3,Ridge Regression,SOH,8.276790e-01,7.763909e-01,1.002516e+00,0.988268
